# echoes — recommendation engine
### `03_recommender_dev.ipynb`

Building a hybrid music recommendation engine with three layers:
- **Layer 1** — Content-based filtering (TF-IDF cosine similarity)
- **Layer 2** — Collaborative filtering (item-item, Last.fm similarity network)
- **Layer 3** — Hybrid ranker (weighted blend α=0.4, β=0.6)

All models are built on real Last.fm listening data fetched in `02_eda.ipynb`.

---

## 0. setup & data loading
Load precomputed assets from EDA — history, clusters, TF-IDF matrix.

In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from utils.lastfm import get_similar_artists, get_artist_tags

load_dotenv('/Users/saturnine/echoes/.env', override=True)

print("✓ Environment ready")
print("✓ Key loaded:", os.getenv("LASTFM_API_KEY")[:6] + "...")

✓ Environment ready
✓ Key loaded: 24381e...


In [2]:
history     = pd.read_csv('/Users/saturnine/echoes/data/history.csv')
top_artists = pd.read_csv('/Users/saturnine/echoes/data/top_artists.csv')
top_tracks  = pd.read_csv('/Users/saturnine/echoes/data/top_tracks.csv')
cluster_df  = pd.read_csv('/Users/saturnine/echoes/data/artist_clusters.csv')
matrix_norm = np.load('/Users/saturnine/echoes/data/tfidf_final.npy')

with open('/Users/saturnine/echoes/data/artists_final.json') as f:
    artists_final = json.load(f)

with open('/Users/saturnine/echoes/data/artist_tags_final.json') as f:
    artist_tags_final = json.load(f)

history['datetime'] = pd.to_datetime(history['datetime'])
USERNAME = "thesaturnineeee"

print(f"✓ History:       {len(history):,} tracks")
print(f"✓ Top artists:   {len(top_artists)} artists")
print(f"✓ Top tracks:    {len(top_tracks)} tracks")
print(f"✓ Clusters:      {cluster_df['cluster'].nunique()} clusters, {len(cluster_df)} artists")
print(f"✓ TF-IDF matrix: {matrix_norm.shape}")

✓ History:       4,974 tracks
✓ Top artists:   20 artists
✓ Top tracks:    50 tracks
✓ Clusters:      5 clusters, 48 artists
✓ TF-IDF matrix: (48, 80)


## 1. content-based filtering

**Idea:** represent each artist as a TF-IDF vector of their Last.fm tags.
Find artists similar to your top artists using cosine similarity.

**Why this works:** artists you love share tags like `dream pop`, `sadcore`, `indie` —
cosine similarity finds others with the same tag fingerprint.

**Note:** raw tags had noise and missing values — we enrich thin profiles
and apply a refined noise filter before building the final TF-IDF matrix.

In [3]:
# artist index lookups
artist_to_idx = {artist: i for i, artist in enumerate(artists_final)}
idx_to_artist = {i: artist for i, artist in enumerate(artists_final)}

# tag enrichment for artists with thin Last.fm profiles
TAG_ENRICHMENT = {
    'Pritam':             ['bollywood', 'hindi', 'soundtrack', 'romantic', 'melody'],
    'A.R. Rahman':        ['bollywood', 'soundtrack', 'indian', 'orchestral', 'melodic'],
    'Shankar Mahadevan':  ['bollywood', 'hindi', 'classical', 'devotional'],
    'Siddharth Pandit':   ['hindi', 'bollywood', 'romantic', 'melody'],
    'DIGV':               ['hindi', 'indie', 'bollywood', 'romantic'],
    'Ana Rehman':         ['pakistani', 'soft rock', 'romantic', 'acoustic'],
    'Prithvi Gandharv':   ['hindi', 'indie', 'romantic', 'folk'],
    'Amit Sharma':        ['hindi', 'bollywood', 'romantic', 'melody'],
    'Mohit Chauhan':      ['hindi', 'bollywood', 'indie', 'acoustic', 'romantic'],
    'Rochak Kohli':       ['hindi', 'bollywood', 'romantic', 'pop'],
    'Armaan Malik':       ['hindi', 'bollywood', 'pop', 'romantic'],
    'Salim-Sulaiman':     ['bollywood', 'hindi', 'electronic', 'fusion'],
    'Darshan Raval':      ['hindi', 'indie', 'romantic', 'pop'],
    'Harshit Saxena':     ['hindi', 'bollywood', 'classical', 'devotional'],
    'Ankit Tiwari':       ['hindi', 'bollywood', 'romantic', 'pop'],
    'Rashid Ali':         ['hindi', 'bollywood', 'sufi', 'classical'],
    'Karthik':            ['bollywood', 'hindi', 'playback', 'melodic'],
    'Kushagra':           ['metal', 'experimental', 'rock', 'alternative'],
    'Mitraz':             ['indie', 'hindi', 'pop', 'acoustic', 'chill'],
}

# merge original + enriched tags
artist_tags_enriched = {}
for artist in artists_final:
    existing = artist_tags_final.get(artist, [])
    extra    = TAG_ENRICHMENT.get(artist, [])
    artist_tags_enriched[artist] = list(dict.fromkeys(existing + extra))

# noise tags — artist names, nationalities, meaningless labels
NOISE_REFINED = {
    'all', 'music', 'pritam', 'shankar', 'rahman', 'atif',
    'himesh', 'kailash', 'abhijeet', 'buddamat', 'enrique',
    'kot', 'ost', 'bollybolly', 'ofwgkta', 'bangla',
    'shankar mahadevan', 'shankar ehsaan loy', 'amit trivedi',
    'tollywood', 'mondiovision', 'radio bombay',
    "can't spell isai without sai", 'goat abhyankkar',
    'snake', 'taylor swift', 'a r rahman', 'mohit chauhan',
    'dev d', 'udaan', 'golden', 'steampunk', 'bharat',
    'bollywood soundtrack', 'film music composer',
    'bollywood film', 'desi filmi', 'indian music director',
}

# build clean TF-IDF matrix
corpus = []
for artist in artists_final:
    tags = artist_tags_enriched.get(artist, [])
    tags = [t.lower() for t in tags
            if t.lower() not in NOISE_REFINED and len(t) > 2]
    corpus.append(" ".join(tags) if tags else "pop")

tfidf_cb    = TfidfVectorizer(max_features=120, ngram_range=(1, 1), min_df=1)
matrix_cb   = normalize(tfidf_cb.fit_transform(corpus).toarray())
sim_matrix  = cosine_similarity(matrix_cb)

print(f"✓ TF-IDF matrix: {matrix_cb.shape}")
print(f"✓ Similarity matrix: {sim_matrix.shape}")

def get_similar_artists_cb(artist_name, n=10):
    """Content-based: cosine similarity on TF-IDF tag vectors."""
    if artist_name not in artist_to_idx:
        return []
    idx        = artist_to_idx[artist_name]
    sim_scores = sorted(enumerate(sim_matrix[idx]),
                        key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i != idx and s > 0]
    return [(idx_to_artist[i], round(s, 4)) for i, s in sim_scores[:n]]

# validation
print("\n=== CONTENT-BASED SIMILARITY ===\n")
for artist in ['Lana Del Rey', 'Pritam', 'The Smiths', 'Atif Aslam', 'A.R. Rahman']:
    similar = get_similar_artists_cb(artist, n=5)
    print(f"{artist}:")
    for name, score in similar:
        tags = [t for t in artist_tags_enriched.get(name, [])
                if t.lower() not in NOISE_REFINED][:3]
        print(f"  {score:.3f}  {name:<25} {tags}")
    print()

✓ TF-IDF matrix: (48, 96)
✓ Similarity matrix: (48, 48)

=== CONTENT-BASED SIMILARITY ===

Lana Del Rey:
  0.438  Taylor Swift              ['country', 'pop', 'female vocalists']
  0.430  Darshan Raval             ['indie', 'hindi', 'romantic']
  0.408  Himesh Reshammiya         []
  0.408  Abhijeet                  []
  0.363  The Smiths                ['post-punk', 'indie', 'new wave']

Pritam:
  0.483  Arijit Singh              ['indipop', 'romantic', 'melody']
  0.442  Rochak Kohli              ['playback', 'hindi', 'bollywood']
  0.439  Ankit Tiwari              ['melody', 'downtempo', 'love song']
  0.423  Armaan Malik              ['ballad', 'hindi', 'bollywood']
  0.418  Mohit Chauhan             ['hindi', 'bollywood', 'indie']

The Smiths:
  0.598  Anirudh Ravichander       ['pop', 'rock', 'alternative']
  0.384  Darshan Raval             ['indie', 'hindi', 'romantic']
  0.370  Mitraz                    ['chill', 'indie', 'hindie']
  0.363  Lana Del Rey              ['indie', 

## 2. collaborative filtering — item-item approach

**Why not user-user?**  
Last.fm's API restricts access to other users' listening data —
a real-world constraint every recommendation engineer faces.

**Solution: item-item collaborative filtering**  
Use `artist.getSimilar` — Last.fm's own collaborative signal,
built from millions of co-listening events across their entire user base.
This is stronger than a small user-user matrix.

**Pipeline:**
1. Take your top artists as seeds, weighted by playcount
2. Fetch Last.fm similar artists for each (collaborative signal)
3. Score candidates by how many seed artists recommend them
4. Weight by playcount → artists you love most have more influence

In [4]:
def get_collaborative_recommendations(top_n_sources=10, candidates_per=10):
    """
    Item-item collaborative filtering using Last.fm's
    artist similarity network (built from millions of users).

    Score = sum of (source_artist_playcount_weight × 1)
    for every candidate across all seed artists.
    """
    seeds         = top_artists.head(top_n_sources)[['name','playcount']].copy()
    seeds['weight'] = seeds['playcount'] / seeds['playcount'].sum()
    known_artists = set(history['artist'].str.lower().unique())

    candidate_scores  = {}
    candidate_sources = {}

    print(f"Building collaborative recommendations from {len(seeds)} seed artists...\n")

    for _, row in seeds.iterrows():
        source  = row['name']
        weight  = row['weight']
        similar = get_similar_artists(source, limit=candidates_per)
        print(f"  {source:<30} ({row['playcount']} plays) → {len(similar)} similar")

        for candidate in similar:
            if candidate.lower() in known_artists:
                continue
            if candidate not in candidate_scores:
                candidate_scores[candidate]  = 0
                candidate_sources[candidate] = []
            candidate_scores[candidate] += weight
            candidate_sources[candidate].append(source)
        time.sleep(0.2)

    results = pd.DataFrame([
        {'artist': a, 'collab_score': round(s, 4),
         'recommended_by': candidate_sources[a],
         'source_count': len(candidate_sources[a])}
        for a, s in candidate_scores.items()
    ])

    results = results.sort_values(
        ['source_count', 'collab_score'], ascending=False
    ).reset_index(drop=True)

    print(f"\n✓ Total candidate artists: {len(results)}")
    print(f"✓ Recommended by 2+ sources: {len(results[results['source_count'] >= 2])}")
    return results

collab_recs = get_collaborative_recommendations()

print("\n=== TOP COLLABORATIVE RECOMMENDATIONS ===\n")
for _, row in collab_recs.head(15).iterrows():
    print(f"  {row['artist']:<30} score: {row['collab_score']:.3f}  "
          f"sources: {row['source_count']}  via: {row['recommended_by'][:2]}")

Building collaborative recommendations from 10 seed artists...

  Lana Del Rey                   (8050 plays) → 10 similar
  Arctic Monkeys                 (7114 plays) → 10 similar
  Taylor Swift                   (2895 plays) → 10 similar
  Frank Ocean                    (2811 plays) → 10 similar
  The Weeknd                     (2199 plays) → 10 similar
  Kendrick Lamar                 (1634 plays) → 10 similar
  The Last Shadow Puppets        (1624 plays) → 10 similar
  Tame Impala                    (1572 plays) → 10 similar
  The Smiths                     (1488 plays) → 10 similar
  Pritam                         (1367 plays) → 10 similar

✓ Total candidate artists: 68
✓ Recommended by 2+ sources: 7

=== TOP COLLABORATIVE RECOMMENDATIONS ===

  Lorde                          score: 0.356  sources: 2  via: ['Lana Del Rey', 'Taylor Swift']
  Alex Turner                    score: 0.284  sources: 2  via: ['Arctic Monkeys', 'The Last Shadow Puppets']
  Miles Kane                     

## 3. hybrid ranker

**Idea:** combine content-based and collaborative scores into one final ranking.

**Formula:**
```
final_score = α × content_score + β × collab_score
              α=0.4              β=0.6
```

**Why these weights?**  
Collaborative signal (β=0.6) is weighted higher because Last.fm's similarity
network is built from millions of real listening events — it's a stronger signal
than our 48-artist TF-IDF index. Content-based (α=0.4) adds tag-level precision.
These weights are validated in `04_ab_testing.ipynb`.

**External candidate pool:**  
Since our TF-IDF index only covers your 48 known artists,
we first fetch external candidates via Last.fm, then score them
in a unified TF-IDF space that includes both your artists and the candidates.

In [5]:
def build_external_candidate_pool(top_n_sources=8, candidates_per=15):
    """Fetch external artists not in your library as recommendation candidates."""
    known      = set(history['artist'].str.lower().unique())
    candidates = {}

    print("Building external candidate pool...")
    for _, row in top_artists.head(top_n_sources).iterrows():
        similar = get_similar_artists(row['name'], limit=candidates_per)
        new     = [a for a in similar if a.lower() not in known]
        print(f"  {row['name']:<30} → {len(new)} new candidates")
        for artist in new:
            if artist not in candidates:
                candidates[artist] = []
        time.sleep(0.2)

    print(f"\n✓ Total external candidates: {len(candidates)}")
    print("Fetching tags for candidates...")

    for i, artist in enumerate(list(candidates.keys())):
        candidates[artist] = get_artist_tags(artist, limit=10)
        if i % 10 == 0 and i > 0:
            print(f"  {i}/{len(candidates)} tags fetched...")
        time.sleep(0.15)

    print(f"✓ Tags fetched for all {len(candidates)} candidates")
    return candidates

external_candidates = build_external_candidate_pool()

print("\nSample candidates with tags:")
for artist, tags in list(external_candidates.items())[:8]:
    print(f"  {artist:<30} {tags[:4]}")

Building external candidate pool...
  Lana Del Rey                   → 10 new candidates
  Arctic Monkeys                 → 13 new candidates
  Taylor Swift                   → 12 new candidates
  Frank Ocean                    → 13 new candidates
  The Weeknd                     → 11 new candidates
  Kendrick Lamar                 → 14 new candidates
  The Last Shadow Puppets        → 13 new candidates
  Tame Impala                    → 14 new candidates

✓ Total external candidates: 81
Fetching tags for candidates...
  10/81 tags fetched...
  20/81 tags fetched...
  30/81 tags fetched...
  40/81 tags fetched...
  50/81 tags fetched...
  60/81 tags fetched...
  70/81 tags fetched...
  80/81 tags fetched...
✓ Tags fetched for all 81 candidates

Sample candidates with tags:
  Emile Haynie                   ['indie', 'experimental', 'pop', 'alternative']
  Melanie Martinez               ['pop', 'indie', 'indie pop', 'female vocalists']
  Lorde                          ['pop', 'indie pop'

In [6]:
def score_external_candidates(external_candidates, top_n=20):
    """
    Score external candidates by:
    1. Content score  — tag cosine similarity to your top artists
    2. Collab score   — how many of your top artists recommend them
    3. Final score    — α × content + β × collab
    """
    # build unified TF-IDF space (your artists + candidates)
    all_tag_strings = []
    for artist in artists_final:
        tags = artist_tags_enriched.get(artist, [])
        tags = [t for t in tags if t.lower() not in NOISE_REFINED and len(t) > 2]
        all_tag_strings.append(" ".join(tags) if tags else "pop")
    for artist, tags in external_candidates.items():
        clean = [t for t in tags if len(t) > 2]
        all_tag_strings.append(" ".join(clean) if clean else "pop")

    tfidf_u  = TfidfVectorizer(max_features=150, ngram_range=(1, 1))
    matrix_u = normalize(tfidf_u.fit_transform(all_tag_strings).toarray())

    your_matrix      = matrix_u[:len(artists_final)]
    candidate_matrix = matrix_u[len(artists_final):]

    # content scores — weighted cosine similarity to your top artists
    top10_artists = top_artists['name'].head(10).tolist()
    top10_plays   = top_artists['playcount'].head(10).values
    valid_pairs   = [(a, p) for a, p in zip(top10_artists, top10_plays)
                    if a in artists_final]
    valid_idx     = [artists_final.index(a) for a, p in valid_pairs]
    valid_weights = np.array([p for a, p in valid_pairs])
    valid_weights = valid_weights / valid_weights.sum()

    print(f"✓ Seed artists for CB scoring: {len(valid_pairs)}")
    cb_scores_raw = cosine_similarity(candidate_matrix,
                                      your_matrix[valid_idx]) @ valid_weights

    # collab scores — how many of your top artists recommend each candidate
    cf_scores_raw  = []
    cf_sources_map = {}
    for artist in external_candidates.keys():
        score, sources = 0, []
        for _, row in top_artists.head(10).iterrows():
            similar = get_similar_artists(row['name'], limit=15)
            weight  = row['playcount'] / top_artists['playcount'].sum()
            if artist in similar:
                score += weight
                sources.append(row['name'])
        cf_scores_raw.append(score)
        cf_sources_map[artist] = sources
    cf_scores_raw = np.array(cf_scores_raw)

    # normalize to 0-1
    cb_norm = cb_scores_raw / (cb_scores_raw.max() + 1e-9)
    cf_norm = cf_scores_raw / (cf_scores_raw.max() + 1e-9)

    # hybrid blend
    alpha, beta  = 0.4, 0.6
    final_scores = alpha * cb_norm + beta * cf_norm

    results = []
    for i, artist in enumerate(external_candidates.keys()):
        cb, cf, fin = float(cb_norm[i]), float(cf_norm[i]), float(final_scores[i])
        total = cb + cf + 1e-9
        if cb > 0.1 and cf > 0.1:
            reason = "tag match + listener overlap"
        elif cb > cf:
            reason = "tag similarity"
        else:
            reason = "listener overlap"
        results.append({
            'artist':        artist,
            'final_score':   round(fin, 4),
            'content_score': round(cb, 4),
            'collab_score':  round(cf, 4),
            'cb_pct':        round(cb / total * 100),
            'cf_pct':        round(cf / total * 100),
            'reason':        reason,
            'via':           cf_sources_map.get(artist, [])[:2],
            'tags':          external_candidates[artist][:4],
        })

    results_df = pd.DataFrame(results).sort_values(
        'final_score', ascending=False
    ).reset_index(drop=True)
    return results_df.head(top_n)

print("Scoring candidates... (takes ~2 mins)")
hybrid_recs = score_external_candidates(external_candidates, top_n=20)

print("\n=== HYBRID RECOMMENDATIONS ===\n")
print(f"{'artist':<30} {'score':>6}  {'cb%':>4}  {'cf%':>4}  {'reason':<25} tags")
print("─" * 90)
for _, row in hybrid_recs.iterrows():
    print(f"  {row['artist']:<28} {row['final_score']:>6.3f}  "
          f"{row['cb_pct']:>3}%  {row['cf_pct']:>3}%  "
          f"{row['reason']:<25} {row['tags'][:3]}")

Scoring candidates... (takes ~2 mins)
✓ Seed artists for CB scoring: 5

=== HYBRID RECOMMENDATIONS ===

artist                          score   cb%   cf%  reason                    tags
──────────────────────────────────────────────────────────────────────────────────────────
  Alexandra Savior              0.879   56%   44%  tag match + listener overlap ['indie pop', 'singer-songwriter', 'indie']
  Lorde                         0.837   37%   63%  tag match + listener overlap ['pop', 'indie pop', 'electronic']
  Madison Beer                  0.821   36%   64%  tag match + listener overlap ['pop', 'female vocalists', 'rnb']
  Marina & the Diamonds         0.745   51%   49%  tag match + listener overlap ['female vocalists', 'indie pop', 'indie']
  Albert Hammond, Jr.           0.730   44%   56%  tag match + listener overlap ['indie', 'indie rock', 'singer-songwriter']
  Melanie Martinez              0.709   48%   52%  tag match + listener overlap ['pop', 'indie', 'indie pop']
  Zella Day

## 4. explainability layer

Every recommendation gets a human-readable explanation showing:
- Which of your artists triggered this recommendation
- What tags match your taste profile
- The score breakdown between content and collaborative signals

This is what separates a good recommender from a great one.
See `04_ab_testing.ipynb` for statistical validation.

In [7]:
def explain_recommendation(artist, hybrid_recs, external_candidates):
    """Generate human-readable explanation for a recommendation."""
    row = hybrid_recs[hybrid_recs['artist'] == artist]
    if row.empty:
        return f"No explanation found for {artist}"
    row  = row.iloc[0]
    tags = external_candidates.get(artist, [])[:5]

    your_tags    = set()
    for a in artists_final:
        your_tags.update(artist_tags_enriched.get(a, []))
    matching_tags = [t for t in tags if t.lower() in {t2.lower() for t2 in your_tags}]

    return "\n".join([
        f"artist:         {artist}",
        f"score:          {row['final_score']:.3f}",
        f"tags:           {', '.join(tags)}",
        f"matching tags:  {', '.join(matching_tags) if matching_tags else 'genre similarity'}",
        f"recommended by: {', '.join(row['via']) if row['via'] else 'tag similarity'}",
        f"breakdown:      {row['cb_pct']}% content  |  {row['cf_pct']}% collaborative",
    ])

print("=== RECOMMENDATION EXPLANATIONS ===\n")
for artist in hybrid_recs['artist'].head(5):
    print(explain_recommendation(artist, hybrid_recs, external_candidates))
    print()

# save final recommendations
hybrid_recs.to_csv('/Users/saturnine/echoes/data/hybrid_recs.csv', index=False)
print("✓ Recommendations saved to data/hybrid_recs.csv")

=== RECOMMENDATION EXPLANATIONS ===

artist:         Alexandra Savior
score:          0.879
tags:           indie pop, singer-songwriter, indie, indie rock, female vocalists
matching tags:  indie pop, singer-songwriter, indie, indie rock, female vocalists
recommended by: Arctic Monkeys, The Last Shadow Puppets
breakdown:      56% content  |  44% collaborative

artist:         Lorde
score:          0.837
tags:           pop, indie pop, electronic, indie, new zealand
matching tags:  pop, indie pop, electronic, indie
recommended by: Lana Del Rey, Taylor Swift
breakdown:      37% content  |  63% collaborative

artist:         Madison Beer
score:          0.821
tags:           pop, female vocalists, rnb, indie, indie pop
matching tags:  pop, female vocalists, rnb, indie, indie pop
recommended by: Lana Del Rey, Taylor Swift
breakdown:      36% content  |  64% collaborative

artist:         Marina & the Diamonds
score:          0.745
tags:           female vocalists, indie pop, indie, british